In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("content/flipkart_product_review.csv")
data.head()

### **Convert data into Document format**

In [ ]:
data = data[['product_title' , 'review']]
data.head()

In [ ]:
product_list = []

for index,row in data.iterrows():
    # Making a temp object
    object = {
        "product_name" : row["product_title"],
        "review" : row['review']
    }
    # Append the object to the product list
    product_list.append(object)

print(product_list[0])

In [ ]:
from langchain_core.documents import Document
docs = []

for object in product_list:
    metadata = {"product_name":object['product_name']}
    page_content = object['review']
    doc = Document(page_content=page_content , metadata = metadata)
    docs.append(doc)

In [ ]:
docs[0]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

hugging_face_api = os.getenv("HF_TOKEN")

### **Store Embeddings to the Astra**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

In [ ]:
astra_db_end_point = os.getenv("astra_db_end_point")
astra_db_app_token = os.getenv("astra_db_token")


In [ ]:
print("Uploading documents and generating embeddings... Please wait.")

from langchain_astradb import AstraDBVectorStore

vector_store = AstraDBVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    api_endpoint=astra_db_end_point,
    token=astra_db_app_token,
    collection_name="flipkart",
    namespace=os.getenv("astra_db_keyspace")  # Maps to your keyspace name
)

### **LLM WORK**

In [ ]:
from langchain_groq import ChatGroq
groq_api = os.getenv("groq_api")
model = ChatGroq(api_key=groq_api , model = "llama-3.3-70b-versatile")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# --- 1. Set up  Prompts & Retriever ---
retriever_prompt = (
    "Given a chat history and the latest user question which might reference context in the chat history, "
    "formulate a standalone question which can be understood without the chat history. "
    "Do NOT answer the question, just reformulate it if needed and otherwise return it as is."
)

contexulized_q_prompt = ChatPromptTemplate.from_messages([
    ('system' , retriever_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human' , '{input}')
])

PRODUCT_BOT_TEMPLATE = """
Your ecommercebot bot is an expert in product recommendations and customer queries.
It analyzes product titles and reviews to provide accurate and helpful responses.
Ensure your answers are relevant to the product context and refrain from straying off-topic.
Your responses should be concise and informative.

CONTEXT:
{context}

QUESTION: {input}

YOUR ANSWER:
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system' , PRODUCT_BOT_TEMPLATE),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human' , '{input}')
])


retriver = vector_store.as_retriever(search_kwargs = {'k' : 3})


# Custom History aware Retriver
condense_question_chain = contexulized_q_prompt | model | StrOutputParser()


def retrive_documents(inputs):
    # If there is chat history then condensed question chain first
    if inputs.get("chat_history"):
        standalone_question = condense_question_chain.invoke({
            "chat_history": inputs["chat_history"],
            "input": inputs["input"]
        })
    else:
        standalone_question = inputs['input']

    return retriver.invoke(standalone_question)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    # Step 1: Fetch and add the context to the input payload
    RunnablePassthrough.assign(
        context=RunnableLambda(retrive_documents) | format_docs
    )
    # Step 2: Pass everything to the prompt/model, and pack the result into an "answer" key
    | RunnablePassthrough.assign(
        answer=qa_prompt | model | StrOutputParser()  # <-- Wraps the string response into {"answer": "..."}
    )
)

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer", # Adjust based on whether your final model execution returns a string or object
)

In [ ]:
chain_with_memory.invoke(
   {"input": "can you tell me the best bluetooth buds?"},
    config={
        "configurable": {"session_id": "Paras"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

In [ ]:
store 

In [ ]:
chain_with_memory.invoke(
   {"input": "what is my previous question?"},
    config={
        "configurable": {"session_id": "Paras"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

In [ ]:
store